# Exploratory Data Analysis - Installments Payments Dataset

EDA on `installments_payments.csv` from the Home Credit Default Risk competition: the repayment ledger for a client's previous Home Credit loans — one row per observed payment on an installment (a single installment can have several payment rows).

**Related Notebooks:**
- `application_eda.ipynb` - Application_train dataset
- `bureau_eda.ipynb` - Bureau dataset
- `previous_application_eda.ipynb` - Previous applications
- `credit_card_balance_eda.ipynb` - Credit card balances

**Analysis Sections:**
1. Setup and Data Loading
2. Basic Dataset Exploration
3. Missing Data Analysis
4. Numeric Features Analysis
5. Categorical Features Analysis
6. Standard Data Quality Checks
7. Functional / Business-Logic Quality Checks

## 1. Setup and Data Loading

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl
import seaborn as sns

# Add src to path to import custom modules
sys.path.insert(0, "/Users/bharathregula/code/home_credit_risk_deeplearn/src")
from home_credit_risk.eda_utils import (
    analyze_categorical_features,
    analyze_numeric_features,
    check_categorical_placeholders,
    check_data_quality,
    check_duplicates,
    detect_sentinel_values,
    display_basic_info,
    show_missing_patterns,
)

# Configure polars display options
pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(-1)  # show all columns
pl.Config.set_fmt_str_lengths(80)

# Set style for plots
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

In [ ]:
# Load dataset
datasets = Path("/Users/bharathregula/code/datasets/home-credit-default-risk")

# Every check below reuses the whole frame, so we collect it once. A large
# infer_schema_length guards against dtype mis-inference across the columns.
inst_df = pl.read_csv(
    datasets / "installments_payments.csv", infer_schema_length=100000
)

print("Dataset loaded successfully!")
print(f"Shape: {inst_df.shape}")

In [ ]:
# Quick preview of the data
inst_df.head()

## 2. Basic Dataset Exploration

In [ ]:
# Display basic dataset information
display_basic_info(inst_df, df_name="Installments_Payments")

In [ ]:
# Summary statistics
inst_df.describe()

## 3. Missing Data Analysis

The natural key `(SK_ID_PREV, NUM_INSTALMENT_VERSION, NUM_INSTALMENT_NUMBER)` is **not** unique — an installment can be paid across several rows — so full-row duplicates are checked here and the key cardinality is examined in the functional section.

In [ ]:
# Analyze missing data patterns
missing_summary = show_missing_patterns(inst_df, threshold=0.0)

In [ ]:
# Check for full-row duplicate records
check_duplicates(inst_df)

## 4. Numeric Features Analysis

In [ ]:
# Analyze numeric features
numeric_df = analyze_numeric_features(inst_df, figsize=(18, 6), show_summary=True)

## 5. Categorical Features Analysis

This table is entirely numeric, so this is expected to report no categorical columns — included for consistency with the other EDA notebooks.

In [ ]:
# Analyze categorical features
analyze_categorical_features(inst_df)

## 6. Standard Data Quality Checks

In [ ]:
# Generic data quality checks
check_data_quality(inst_df)

## 7. Functional / Business-Logic Quality Checks

Repayment-behaviour rules: payment **presence** consistency, **timeliness** (paid vs due date) and **completeness** (paid vs prescribed amount). Timeliness and completeness are also strong behavioural signals for default modelling.

In [ ]:
# Small helper for consistent count + percentage reporting
n = inst_df.height


def pct(count: int) -> str:
    return f"{count:,} ({count / n * 100:.3f}%)"


print(f"Total installment payment rows: {n:,}")

### A. Grain / key cardinality

The key can repeat (partial payments), so we quantify how many rows share a key rather than asserting uniqueness. `DAYS_*` are relative to the current application, so both must be ≤ 0.

In [ ]:
# A1. Rows sharing a (SK_ID_PREV, VERSION, NUMBER) key -> multiple payment entries
grain_cols = ["SK_ID_PREV", "NUM_INSTALMENT_VERSION", "NUM_INSTALMENT_NUMBER"]
grain_dupes = n - inst_df.select(grain_cols).n_unique()
print(
    f"A1. Rows sharing an installment key (multi-row installments): {pct(grain_dupes)}"
)

# A2. DAYS columns must be <= 0 (past, relative to application)
a2_inst = inst_df.filter(pl.col("DAYS_INSTALMENT") > 0).height
a2_pay = inst_df.filter(pl.col("DAYS_ENTRY_PAYMENT") > 0).height
print(f"A2. DAYS_INSTALMENT > 0: {pct(a2_inst)}")
print(f"    DAYS_ENTRY_PAYMENT > 0: {pct(a2_pay)}")

### B. Payment presence consistency

A missing payment should null **both** `DAYS_ENTRY_PAYMENT` and `AMT_PAYMENT` together — exactly one being null is inconsistent.

In [ ]:
# B1. DAYS_ENTRY_PAYMENT and AMT_PAYMENT must be null together
entry_null = pl.col("DAYS_ENTRY_PAYMENT").is_null()
amt_null = pl.col("AMT_PAYMENT").is_null()
b1_one = inst_df.filter(entry_null != amt_null).height
b1_both = inst_df.filter(entry_null & amt_null).height
print(f"B1. Exactly one of (DAYS_ENTRY_PAYMENT, AMT_PAYMENT) null: {pct(b1_one)}")
print(f"    Both null (installment never paid): {pct(b1_both)}")

### C. Payment timeliness

A payment entered after its due date is **late** (`DAYS_ENTRY_PAYMENT > DAYS_INSTALMENT`). The late rate and how-late distribution are behavioural default signals.

In [ ]:
# C1. Late payments
paid = inst_df.filter(pl.col("DAYS_ENTRY_PAYMENT").is_not_null())
late = paid.filter(pl.col("DAYS_ENTRY_PAYMENT") > pl.col("DAYS_INSTALMENT"))
print(f"C1. Late payments (of {paid.height:,} with a payment): {pct(late.height)}")
days_late = late.select(
    (pl.col("DAYS_ENTRY_PAYMENT") - pl.col("DAYS_INSTALMENT")).alias("days_late")
)["days_late"]
print(f"    days late when late -> mean {days_late.mean():.1f}, max {days_late.max()}")

### D. Payment completeness

The paid amount vs the prescribed installment: underpayment, overpayment, or a zero payment against a non-zero installment.

In [ ]:
# D1. Under / over / zero payment (tolerance 0.01)
TOL = 0.01
has_pay = inst_df.filter(pl.col("AMT_PAYMENT").is_not_null())
under = has_pay.filter(pl.col("AMT_PAYMENT") < pl.col("AMT_INSTALMENT") - TOL).height
over = has_pay.filter(pl.col("AMT_PAYMENT") > pl.col("AMT_INSTALMENT") + TOL).height
zero = has_pay.filter(
    (pl.col("AMT_PAYMENT") == 0) & (pl.col("AMT_INSTALMENT") > 0)
).height
print(f"D1. Underpaid: {pct(under)}")
print(f"    Overpaid : {pct(over)}")
print(f"    Zero payment on a due installment: {pct(zero)}")

### E. Non-negativity / bounds

Amounts must be ≥ 0 and the installment index must be ≥ 1.

In [ ]:
# E1. Non-negativity / bounds
bound_checks = [
    ("AMT_INSTALMENT < 0", pl.col("AMT_INSTALMENT") < 0),
    ("AMT_PAYMENT < 0", pl.col("AMT_PAYMENT") < 0),
    ("NUM_INSTALMENT_NUMBER < 1", pl.col("NUM_INSTALMENT_NUMBER") < 1),
]
for label, expr in bound_checks:
    print(f"E1. {label}: {pct(inst_df.filter(expr).height)}")

### F. Sentinel / placeholder detection

Reuse the shared helpers to confirm there is no `365243` numeric sentinel or `XNA`/`XAP` placeholder (this all-numeric table is expected to be clean).

In [ ]:
# F1. Numeric sentinel + categorical placeholders.
# detect_sentinel_values auto-skips SK_ID* identifier columns, so an id that
# happens to equal 365243 is not reported as a false positive.
sentinel_summary = detect_sentinel_values(inst_df)
placeholder_summary = check_categorical_placeholders(inst_df)